[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter2/sqlite-vec.ipynb)

# Chapter 2: SQLite Vector Search with sqlite-vec

Not every RAG application needs a dedicated vector database server. For prototyping, edge deployments, or small-scale applications, SQLite with the sqlite-vec extension provides an embedded, zero-configuration alternative to server-based solutions like pgvector.

This notebook demonstrates vector similarity search using SQLite's sqlite-vec extension.

**What you'll learn:**
- Set up SQLite with the sqlite-vec extension for vector storage
- Generate sentence embeddings with SentenceTransformers
- Store and query vectors using SQL syntax
- Perform nearest-neighbor similarity search

**Prerequisites:** `pip install sentence-transformers sqlite-vec`

## Setup

We import the libraries, define constants and helper functions, then load the SentenceTransformer model and open a SQLite database with the sqlite-vec extension for vector operations.

In [1]:
import sqlite3
import struct
from contextlib import closing
from dataclasses import dataclass
from pathlib import Path
from typing import Final

import numpy
import numpy.typing
import sqlite_vec  # type: ignore[import-untyped]  # sqlite-vec ships no type stubs
from sentence_transformers import SentenceTransformer

MODEL_NAME: Final = "all-MiniLM-L6-v2"
DATABASE_PATH: Final = Path("embeddings.db")
NEAREST_CHUNK_COUNT: Final = 1
# The search query selects chunk_id, text, and distance.
SEARCH_RESULT_COLUMN_COUNT: Final = 3

# A single embedding: a 1-D array of float32 values.
type FloatVector = numpy.typing.NDArray[numpy.float32]
# Several embeddings: a 2-D array with one row per text.
type FloatMatrix = numpy.typing.NDArray[numpy.float32]

In [2]:
@dataclass(frozen=True, kw_only=True, slots=True)
class ChunkMatch:
    chunk_id: int
    text: str
    distance: float


def open_vector_database(database_path: Path) -> sqlite3.Connection:
    """Open the SQLite database file and load the sqlite-vec extension into it."""
    database_connection = sqlite3.connect(database_path)

    # enable_load_extension() takes its flag as a positional-only argument.
    database_connection.enable_load_extension(True)  # noqa: FBT003  # positional-only stdlib parameter
    try:
        sqlite_vec.load(database_connection)
    finally:
        # Turn extension loading off again, so later SQL cannot load other extensions.
        database_connection.enable_load_extension(False)  # noqa: FBT003  # positional-only stdlib parameter

    return database_connection


def encode_normalized_texts(
    sentence_model: SentenceTransformer,
    texts: list[str],
) -> FloatMatrix:
    """Return one unit-length float32 embedding row per text."""
    encoded_texts: object = sentence_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    if not isinstance(encoded_texts, numpy.ndarray):
        raise TypeError(
            f"encode() returned {type(encoded_texts).__name__}, expected numpy.ndarray."
        )

    if encoded_texts.dtype != numpy.float32:
        raise TypeError(
            f"encode() returned dtype {encoded_texts.dtype}, expected float32."
        )

    return encoded_texts


def serialize_float32_vector(vector: FloatVector) -> bytes:
    """Pack a 1-D float32 vector into the raw bytes format that sqlite-vec stores.

    struct uses the machine's native byte order. sqlite-vec reads the bytes the same way.
    """
    if vector.ndim != 1:
        raise ValueError(
            f"Expected a 1-D vector, got an array with {vector.ndim} dimensions."
        )

    value_count = len(vector)
    struct_format = f"{value_count}f"
    serialized_vector = struct.pack(struct_format, *vector)

    return serialized_vector


def create_chunks_table(
    database_connection: sqlite3.Connection,
    *,
    embedding_dimension: int,
) -> None:
    """Drop and recreate the `chunks` vec0 table for vectors of the given size."""
    if embedding_dimension <= 0:
        raise ValueError(
            f"Embedding dimension must be positive, got {embedding_dimension}."
        )

    drop_table_sql = "DROP TABLE IF EXISTS chunks"
    # SQLite cannot bind a parameter inside a CREATE statement, so the checked
    # integer dimension is written into the SQL text.
    create_table_sql = (
        "CREATE VIRTUAL TABLE chunks USING vec0("
        "chunk_id INTEGER PRIMARY KEY, "
        "text TEXT, "
        f"embedding float[{embedding_dimension}])"
    )

    # `with database_connection` commits on success and rolls back on error.
    # It does not close the connection.
    with database_connection:
        with closing(database_connection.cursor()) as database_cursor:
            database_cursor.execute(drop_table_sql)
            database_cursor.execute(create_table_sql)


def insert_chunks(
    database_connection: sqlite3.Connection,
    *,
    chunk_texts: list[str],
    chunk_embeddings: FloatMatrix,
) -> None:
    """Insert each text with its embedding. The chunk ID is the text's list position."""
    embedding_row_count = chunk_embeddings.shape[0]

    if len(chunk_texts) != embedding_row_count:
        raise ValueError(
            f"Got {len(chunk_texts)} texts but {embedding_row_count} embedding rows."
        )

    insert_sql = "INSERT INTO chunks (chunk_id, text, embedding) VALUES (?, ?, ?)"

    with database_connection:
        with closing(database_connection.cursor()) as database_cursor:
            for chunk_id, chunk_text in enumerate(chunk_texts):
                chunk_embedding = chunk_embeddings[chunk_id]
                serialized_embedding = serialize_float32_vector(chunk_embedding)
                insert_parameters = (chunk_id, chunk_text, serialized_embedding)
                database_cursor.execute(insert_sql, insert_parameters)


def build_chunk_match(result_row: object) -> ChunkMatch:
    """Check the types in one (chunk_id, text, distance) row and wrap them."""
    if not isinstance(result_row, tuple):
        raise TypeError(f"Expected a tuple row, got {type(result_row).__name__}.")

    if len(result_row) != SEARCH_RESULT_COLUMN_COUNT:
        raise ValueError(
            f"Expected {SEARCH_RESULT_COLUMN_COUNT} columns, got {len(result_row)}."
        )

    chunk_id: object = result_row[0]
    chunk_text: object = result_row[1]
    distance: object = result_row[2]

    if not isinstance(chunk_id, int):
        raise TypeError(f"Column chunk_id must be int, got {type(chunk_id).__name__}.")

    if not isinstance(chunk_text, str):
        raise TypeError(f"Column text must be str, got {type(chunk_text).__name__}.")

    if not isinstance(distance, float):
        raise TypeError(
            f"Column distance must be float, got {type(distance).__name__}."
        )

    chunk_match = ChunkMatch(chunk_id=chunk_id, text=chunk_text, distance=distance)

    return chunk_match


def find_nearest_chunks(
    database_connection: sqlite3.Connection,
    *,
    query_embedding: FloatVector,
    match_count: int,
) -> list[ChunkMatch]:
    """Return the `match_count` chunks closest to the query, nearest first."""
    # sqlite-vec reads `k = ?` as the number of nearest neighbors to return.
    select_sql = (
        "SELECT chunk_id, text, distance "
        "FROM chunks "
        "WHERE embedding MATCH ? AND k = ? "
        "ORDER BY distance"
    )
    serialized_query = serialize_float32_vector(query_embedding)
    select_parameters = (serialized_query, match_count)

    with closing(database_connection.cursor()) as database_cursor:
        database_cursor.execute(select_sql, select_parameters)
        result_rows = database_cursor.fetchall()

    chunk_matches: list[ChunkMatch] = []

    for result_row in result_rows:
        chunk_match = build_chunk_match(result_row)
        chunk_matches.append(chunk_match)

    return chunk_matches


def print_nearest_chunks(
    sentence_model: SentenceTransformer,
    database_connection: sqlite3.Connection,
    *,
    query_text: str,
    match_count: int,
) -> None:
    """Embed the query text, search the `chunks` table, and print each match."""
    query_embeddings = encode_normalized_texts(sentence_model, [query_text])
    query_embedding = query_embeddings[0]

    chunk_matches = find_nearest_chunks(
        database_connection,
        query_embedding=query_embedding,
        match_count=match_count,
    )

    for chunk_match in chunk_matches:
        # The notebook shows search results as cell output.
        print(  # noqa: T201  # notebook display output
            f"Chunk ID: {chunk_match.chunk_id}, "
            f"Text: {chunk_match.text}, "
            f"Distance: {chunk_match.distance}"
        )

In [ ]:
sentence_model = SentenceTransformer(MODEL_NAME)

reported_embedding_dimension = sentence_model.get_sentence_embedding_dimension()

if reported_embedding_dimension is None:
    raise ValueError(f"Model {MODEL_NAME!r} does not report an embedding dimension.")

embedding_dimension: int = reported_embedding_dimension

database_connection = open_vector_database(DATABASE_PATH)

In [ ]:
sentences: list[str] = [
    "I am a happy person.",
    "I am a joyful person.",
    "I am a pessimistic person.",
    "I am not an optimistic person.",
]

sentence_embeddings = encode_normalized_texts(sentence_model, sentences)

## Create Database and Table

We create a virtual table backed by sqlite-vec that stores chunk IDs, text, and fixed-size float vectors in a single schema.

In [ ]:
create_chunks_table(database_connection, embedding_dimension=embedding_dimension)

## Insert Embeddings

Each sentence embedding is serialized to raw bytes and inserted alongside its text so the database can perform vector operations at query time.

In [ ]:
insert_chunks(
    database_connection,
    chunk_texts=sentences,
    chunk_embeddings=sentence_embeddings,
)

## Vector Search

We embed a new query sentence, then use SQL's `MATCH` clause to find the stored vector closest to it — all within a single SQLite database file.

In [ ]:
print_nearest_chunks(
    sentence_model,
    database_connection,
    query_text="I am a glad person.",
    match_count=NEAREST_CHUNK_COUNT,
)

> **Note:** sqlite-vec returns a *distance* (not similarity) — lower values mean closer matches. The query "glad person" matched "happy person" at distance 0.21, confirming that the model recognizes synonyms even when the exact words differ.

In [ ]:
print_nearest_chunks(
    sentence_model,
    database_connection,
    query_text="I am a negative person.",
    match_count=NEAREST_CHUNK_COUNT,
)

## Clean Up

Close the database connection. The `with database_connection:` blocks above only end transactions; they do not close the connection.

In [ ]:
database_connection.close()